In [ ]:
from calitp_data_analysis.gcs_pandas import GCSPandas
from functools import cache

import pandas as pd
import geopandas as gpd
import numpy as np

In [ ]:
@cache
def gcs_pandas():
    return GCSPandas()

In [ ]:
# %env REQUESTS_CA_BUNDLE=C:\Users\s163107\Documents\CTROOTCA01.cer

In [ ]:
culver_tsp_extract = gcs_pandas().read_csv(
    "gs://calitp-analytics-data/data-analyses/tsp-analysis/culver_city_tsp_validation/VehicleState0007140250506_055240.txt",
    skiprows=lambda x: x == 1,
)

### Very basic investigation

In [ ]:
culver_tsp_extract.head()

In [ ]:
list(culver_tsp_extract.columns)

In [ ]:
# See routes present
(culver_tsp_extract.ROUTE_ID.astype(str) + "_" + culver_tsp_extract.TRIP_KEY.astype(str)).value_counts().sort_index()

In [ ]:
# Process time data into python format
culver_tsp_extract["event_time_datetime"] = pd.to_datetime(
    culver_tsp_extract["EVENT_TIME"].astype(str).fillna("000000000000.0"),
    format=r"%y%m%d%H%M%S.0",
)
culver_avl_extract_sorted = culver_tsp_extract.sort_values(["TRIP_KEY", "event_time_datetime"], ascending=True)
culver_avl_extract_sorted[["EVENT_TIME", "event_time_datetime"]].head()
culver_avl_extract_sorted["time_difference"] = (
    culver_avl_extract_sorted.groupby("TRIP_KEY")["event_time_datetime"].diff().dt.total_seconds()
)
culver_avl_extract_sorted["time_difference"].describe()

In [ ]:
# Select a representative trip
trip_1181 = culver_avl_extract_sorted.loc[culver_avl_extract_sorted.TRIP_KEY == 1181]

### Mapping

In [ ]:
# Remove long gaps so the scale works - these are mostly at the start and end of trips
gdf_trip_1181 = gpd.GeoDataFrame(
    trip_1181, geometry=gpd.points_from_xy(trip_1181.LONGITUDE, trip_1181.LATITUDE), crs="EPSG:4326"
).loc[trip_1181.time_difference < 30]
gdf_trip_1181[
    [
        "geometry",
        "time_difference",
        "STOP_BACK_DOOR_ENTRY",
        "STOP_FRONT_DOOR_ENTRY",
        "STOP_FRONT_DOOR_EXIT",
        "STOP_BACK_DOOR_EXIT",
    ]
].explore("time_difference").save("trip_1181.html")

### Frequency Validation

In [ ]:
# Histogram of frequencies for example trip
trip_1181["time_difference"].hist(bins=50)

In [ ]:
# Removing outliers
trip_1181.loc[trip_1181.time_difference < 30, "time_difference"].hist(bins=30)

### Histograms for all trips

In [ ]:
culver_avl_extract_sorted["time_difference"].hist(bins=30)

In [ ]:
# Removing outliers
culver_avl_extract_sorted.loc[culver_avl_extract_sorted.time_difference < 30, "time_difference"].hist(bins=30)

### Exploratory dwell analysis

In [ ]:
DOOR_COLUMNS = [
    "STOP_BACK_DOOR_ENTRY",
    "STOP_FRONT_DOOR_ENTRY",
    "STOP_FRONT_DOOR_EXIT",
    "STOP_BACK_DOOR_EXIT",
]
ANALYSIS_COLUMNS = ["TRIP_KEY", "distance_to_stop", "door_open", "time_difference", *DOOR_COLUMNS, "geometry"]

In [ ]:
# Pick a stop - we'll take overland/washington
analysis_stop = gpd.points_from_xy([-118.404851], [34.017122], crs="EPSG:4326").to_crs("EPSG:3310")[0]

# Get all trip data as a GDF
gdf_culver_avl_extract_sorted = gpd.GeoDataFrame(
    culver_avl_extract_sorted,
    geometry=gpd.points_from_xy(culver_avl_extract_sorted.LONGITUDE, culver_avl_extract_sorted.LATITUDE),
    crs="EPSG:4326",
).dropna(subset=["LONGITUDE", "LATITUDE"])

# Calculate distance to stop and filter to points within 100m
gdf_culver_avl_extract_projected = gdf_culver_avl_extract_sorted.to_crs("EPSG:3310")
gdf_culver_avl_extract_projected["distance_to_stop"] = gdf_culver_avl_extract_projected.geometry.distance(analysis_stop)
gdf_culver_avl_extract_projected["door_open"] = gdf_culver_avl_extract_projected[DOOR_COLUMNS].any(axis=1)
eastbound_culver_avl_extract_near_stop = gdf_culver_avl_extract_projected.loc[
    (gdf_culver_avl_extract_projected.distance_to_stop < 200)
    & (gdf_culver_avl_extract_projected.ROUTE_ID.str.strip() == "104")
]

In [ ]:
sample_trip_near_stop_with_door_open = eastbound_culver_avl_extract_near_stop.loc[
    eastbound_culver_avl_extract_near_stop.TRIP_KEY == 1034
]
sample_trip_near_stop_with_door_open[[*ANALYSIS_COLUMNS, "EVENT_TIME"]].explore("door_open").save(
    "sample_trip_near_stop_with_door_open.html"
)

In [ ]:
# I think it's a little funny how the driver opened the door twice here on each side of the street
sample_trip_near_stop_with_door_open[[*ANALYSIS_COLUMNS]]

### Exploratory signal analysis

In [ ]:
# Pick a signal - we'll use sawtelle/washington
signal_location = gpd.points_from_xy([-118.414594], [34.004383], crs="EPSG:4326").to_crs("EPSG:3310")[0]
gdf_culver_avl_extract_projected["distance_to_signal"] = gdf_culver_avl_extract_projected.geometry.distance(
    signal_location
)
eastbound_culver_avl_extract_near_signal = gdf_culver_avl_extract_projected.loc[
    (gdf_culver_avl_extract_projected.distance_to_signal < 200)
    & (gdf_culver_avl_extract_projected.ROUTE_ID.str.strip() == "104")
].copy()

# Calculate the azimuth between the bus and the signal to see if the bus is approaching or leaving the signal
# lazy code from gemini
x = eastbound_culver_avl_extract_near_signal.geometry.x.values
y = eastbound_culver_avl_extract_near_signal.geometry.y.values
dx = x - signal_location.x
dy = y - signal_location.y
# atan2(dx, dy) gives the angle clockwise from North in radians
azimuths = np.degrees(np.arctan2(dx, dy))
azimuths = (azimuths + 360) % 360
eastbound_culver_avl_extract_near_signal["azimuth_to_signal"] = azimuths
eastbound_culver_avl_extract_ahead_of_signal = eastbound_culver_avl_extract_near_signal.loc[
    eastbound_culver_avl_extract_near_signal.azimuth_to_signal < 120
]

In [ ]:
eastbound_culver_avl_extract_ahead_of_signal[[*ANALYSIS_COLUMNS, "azimuth_to_signal", "EVENT_TIME"]].explore(
    "azimuth_to_signal"
).save("all_trips_near_signal.html")

In [ ]:
# Look at the same trip near the signal - looks like it doesn't stop
sample_trip_no_delay = eastbound_culver_avl_extract_ahead_of_signal.loc[
    eastbound_culver_avl_extract_ahead_of_signal.TRIP_KEY == 1034
]
sample_trip_no_delay[[*ANALYSIS_COLUMNS]].explore("time_difference").save("sample_trip_near_signal.html")

In [ ]:
# It's a little hard to find trips that definitively delay, but this one looks like it does?
sample_trip_with_delay = eastbound_culver_avl_extract_ahead_of_signal.loc[
    eastbound_culver_avl_extract_ahead_of_signal.TRIP_KEY == 1181
]
sample_trip_with_delay[[*ANALYSIS_COLUMNS]].explore("time_difference").save("sample_trip_with_delay_near_signal.html")